# Search-13 (C#) — Recherche à écart limité (Limited Discrepancy Search)

> **Partie 3 — Recherche avancée.** Ce notebook est le **jumeau .NET** du notebook
> [Search-13 — Limited Discrepancy Search](Search-13-LimitedDiscrepancySearch.ipynb)
> (Python). Il implémente en C# le même algorithme — LDS de Harvey & Ginsberg (1995) —
> sur le même banc d'essai (sac à dos 0/1), pour montrer comment la technique se
> transpose à une recherche arborescente typée.
>
> **Prérequis :** avoir vu les fondations de la recherche heuristique en
> [Partie 1 — Search-3 (Informed)](../Part1-Foundations/Search-3-Informed.ipynb)
> (A\*, heuristiques admissibles).


## 2. Le problème-banc d'essai : le sac à dos 0/1

Le **sac à dos binaire** (*0/1 knapsack*) est NP-difficile et se prête naturellement
à une recherche arborescente. On dispose de $n$ objets, chacun avec un poids $w_i$ et
une valeur $v_i$, et d'une capacité $C$. On veut maximiser la valeur totale sans
dépasser la capacité.

L'**heuristique gloutonne** consistant à prendre les objets par **ratio valeur/poids
décroissant** est naturelle mais **myope** : elle peut rater l'optimum en se bloquant
trop tôt. C'est précisément le défaut que LDS corrige.


In [1]:
// Le sac à dos 0/1 : liste d'objets (nom, poids, valeur), ordonnés par RATIO valeur/poids décroissant
// (c'est l'ordre où l'heuristique gloutonne les considère).
// Item: nom, poids, valeur — record C# immuable (équivalent du tuple nommé Python).
public record Item(string Name, int Weight, int Value);

Item[] items = new Item[]
{
    new("A", 6, 8),   // ratio 1.33  <-- top ratio, MAIS poids 6 bloque la paire B+C (valeur 12 > 10)
    new("B", 5, 6),   // ratio 1.20
    new("C", 5, 6),   // ratio 1.20
    new("D", 4, 2),   // ratio 0.50
    new("E", 3, 1),   // ratio 0.33
};
int capacity = 10;

Console.WriteLine($"Objets (ordre heuristique, ratio valeur/poids décroissant) :");
foreach (var it in items)
    Console.WriteLine($"  {it.Name}: poids={it.Weight}, valeur={it.Value}, ratio={it.Value / (double)it.Weight:F2}");
Console.WriteLine($"Capacité du sac : {capacity}");


Objets (ordre heuristique, ratio valeur/poids décroissant) :


  A: poids=6, valeur=8, ratio=1,33


  B: poids=5, valeur=6, ratio=1,20


  C: poids=5, valeur=6, ratio=1,20


  D: poids=4, valeur=2, ratio=0,50


  E: poids=3, valeur=1, ratio=0,33


Capacité du sac : 10


### Stratégie gloutonne (k = 0)

Suivre aveuglément l'heuristique : prendre chaque objet s'il rentre.

In [2]:
// Stratégie GLOUTONNE (k=0) : suivre l'heuristique aveuglement — prendre chaque objet s'il rentre.
(int Value, List<string> Selection, int Nodes) GreedyKnapsack(Item[] items, int capacity)
{
    var sel = new List<string>();
    int totalW = 0, totalV = 0, nodes = 0;
    foreach (var it in items)
    {
        nodes++;
        if (totalW + it.Weight <= capacity)   // heuristique : prendre si ça rentre
        {
            sel.Add(it.Name);
            totalW += it.Weight;
            totalV += it.Value;
        }
    }
    return (totalV, sel, nodes);
}

var (greedyV, greedySel, greedyNodes) = GreedyKnapsack(items, capacity);
Console.WriteLine($"Greedy (k=0) : valeur={greedyV}, sélection=[{string.Join(", ", greedySel)}], nœuds={greedyNodes}");


Greedy (k=0) : valeur=10, sélection=[A, D], nœuds=5


## 3. Le concept d'écart, et l'algorithme LDS(k)

À chaque nœud, l'heuristique **recommande** une action (prendre l'objet s'il rentre,
sinon le laisser). Choisir l'**autre** action coûte **un écart** :

- Objet prenable : rang 0 = **prendre** (recommandé), rang 1 = **laisser** (1 écart).
- Objet non prenable : rang 0 = **laisser** (recommandé), rang 1 = impossible (capacité dépassée).

**LDS(k)** explore tous les chemins de l'arbre qui s'écartent au plus $k$ fois de
l'heuristique. Pour $k = 0$ on retrouve le glouton ; pour $k \to \infty$ on retrouve
la DFS exhaustive. L'intuition de Harvey & Ginsberg : *si l'heuristique est bonne,
l'optimum est un proche voisin du chemin glouton*, donc un petit $k$ suffit souvent.

In [3]:
// LDS(k) sur le sac à dos 0/1 — version récursive bornée par le nombre d'écarts.
// items : tableau ordonné par ordre heuristique (meilleur ratio d'abord).
// Renvoie (meilleure valeur, meilleure sélection, nombre de nœuds explorés).
(int Value, List<string>? Selection, int Nodes) LdsKnapsack(Item[] items, int capacity, int kMax)
{
    int n = items.Length;
    int bestValue = -1;
    List<string>? bestSel = null;
    int nodes = 0;

    void Rec(int i, int remCap, int valSoFar, List<string> sel, int discLeft)
    {
        nodes++;
        if (i == n)
        {
            if (valSoFar > bestValue)
            {
                bestValue = valSoFar;
                bestSel = new List<string>(sel);
            }
            return;
        }
        var it = items[i];
        bool canTake = it.Weight <= remCap;
        // Recommandation de l'heuristique à ce nœud :
        //   - si l'objet rentre : prendre (rang 0)
        //   - sinon : laisser (rang 0)
        // L'autre choix consomme 1 écart (si possible).
        if (canTake)
        {
            // rang 0 (recommandé) : prendre
            sel.Add(it.Name);
            Rec(i + 1, remCap - it.Weight, valSoFar + it.Value, sel, discLeft);
            sel.RemoveAt(sel.Count - 1);
            // rang 1 (écart) : laisser
            if (discLeft > 0)
                Rec(i + 1, remCap, valSoFar, sel, discLeft - 1);
        }
        else
        {
            // rang 0 (recommandé) : laisser (impossible de prendre)
            Rec(i + 1, remCap, valSoFar, sel, discLeft);
            // rang 1 (écart) : prendre — impossible (capacité), branche tronquée
        }
    }

    Rec(0, capacity, 0, new List<string>(), kMax);
    return (bestValue, bestSel, nodes);
}

// Balayage k = 0..n : montre la convergence vers l'optimum.
Console.WriteLine(string.Format("{0,3}{1,8}{2,10}  sélection", "k", "valeur", "nœuds"));
for (int k = 0; k <= items.Length; k++)
{
    var (v, sel, nd) = LdsKnapsack(items, capacity, k);
    Console.WriteLine(string.Format("{0,3}{1,8}{2,10}  [{3}]", k, v, nd, string.Join(", ", sel ?? new List<string>())));
}


  k  valeur     nœuds  sélection


  0      10         6  [A, D]


  1      12        13  [B, C]


  2      12        21  [B, C]


  3      12        28  [B, C]


  4      12        33  [B, C]


  5      12        34  [B, C]



(4,25): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(8,17): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 4. Référence : l'optimum par énumération exhaustive

Pour prouver que LDS atteint bien le **vrai** optimum (et non une borne locale),
on calcule la solution optimale par énumération exhaustive de l'arbre binaire
complet ($2^n$ feuilles). LDS devra converger vers cette même valeur pour $k$ assez grand.

In [4]:
// Optimum exact par énumération exhaustive (2^n) — référence de vérité.
(int Value, List<string> Selection, int Nodes) ExhaustiveKnapsack(Item[] items, int capacity)
{
    int n = items.Length;
    int bestV = -1, nodes = 0;
    List<string> bestSel = new();
    long total = 1L << n;   // 2^n combinaisons
    for (long mask = 0; mask < total; mask++)
    {
        nodes++;
        int w = 0, v = 0;
        var sel = new List<string>();
        for (int i = 0; i < n; i++)
        {
            if (((mask >> i) & 1L) == 1L)
            {
                w += items[i].Weight;
                v += items[i].Value;
                sel.Add(items[i].Name);
            }
        }
        if (w <= capacity && v > bestV)
        {
            bestV = v;
            bestSel = sel;
        }
    }
    return (bestV, bestSel, nodes);
}

var (optV, optSel, exNodes) = ExhaustiveKnapsack(items, capacity);
Console.WriteLine($"Exhaustif : valeur optimale={optV}, sélection=[{string.Join(", ", optSel)}], nœuds={exNodes}");


Exhaustif : valeur optimale=12, sélection=[B, C], nœuds=32


In [5]:
// Tableau comparatif : greedy vs LDS(k) vs exhaustif — QUALITÉ de solution et COÛT (nœuds).
string hdr = string.Format("{0,-18}{1,8}{2,11}{3,10}{4,14}", "stratégie", "valeur", "optimal?", "nœuds", "% exhaustif");
Console.WriteLine(hdr);
Console.WriteLine(new string('-', hdr.Length));

void Row(string label, int val, int nd)
{
    string opt = val == optV ? "OUI" : "non";
    double pct = 100.0 * nd / exNodes;
    Console.WriteLine(string.Format("{0,-18}{1,8}{2,11}{3,10}{4,13:F1}%", label, val, opt, nd, pct));
}

Row("greedy (k=0)", greedyV, greedyNodes);
// LDS au plus petit k atteignant l'optimum
int kStar = -1, ldsVStar = -1, ldsNodesStar = 0;
for (int k = 0; k <= items.Length; k++)
{
    var (v, sel, nd) = LdsKnapsack(items, capacity, k);
    Row($"LDS(k={k})", v, nd);
    if (v == optV && kStar < 0) { kStar = k; ldsVStar = v; ldsNodesStar = nd; }
}
Row("exhaustif", optV, exNodes);
Console.WriteLine();
Console.WriteLine($"=> LDS atteint l'optimum dès k*={kStar}, en {ldsNodesStar} nœuds ({100.0*ldsNodesStar/exNodes:F1}% de l'exhaustif).");


stratégie           valeur   optimal?     nœuds   % exhaustif


-------------------------------------------------------------


greedy (k=0)            10        non         5         15,6%


LDS(k=0)                10        non         6         18,8%


LDS(k=1)                12        OUI        13         40,6%


...

LDS(k=3)                12        OUI        28         87,5%


LDS(k=4)                12        OUI        33        103,1%


LDS(k=5)                12        OUI        34        106,2%


exhaustif               12        OUI        32        100,0%


=> LDS atteint l'optimum dès k*=1, en 13 nœuds (40,6% de l'exhaustif).


### Lecture du résultat

Le **greedy** (k=0) est **sous-optimal** : il prend A au ratio le plus élevé (1,33),
qui remplit 6 des 10 unités de capacité et ne laisse de la place que pour D — ratant
la paire B+C dont la valeur (12) est supérieure. Dès **k=1**, LDS explore le chemin
« laisser A » et découvre l'optimum {B, C, ...}, en n'explorant qu'une fraction de
l'arbre exhaustif. C'est tout l'intérêt de LDS : **saisir l'optimum à coût sous-exponentiel
quand l'heuristique est bonne**.


## 5. Ponts avec le reste de la série

| Direction | Lien | Relation |
|-----------|------|----------|
| ← Fondations | [Search-3 — Informed](../Part1-Foundations/Search-3-Informed.ipynb) | A\*, heuristiques admissibles — LDS suppose une heuristique *ordonnante* (qui range les enfants), notion voisine |
| ↔ Search-12 | [Search-12 — Pattern Databases](Search-12-PatternDatabases.ipynb) | Autre technique avancée pour réduire l'arbre de recherche (mémoïsation de sous-problèmes) |
| ↔ Search-14 | [Search-14 — Weighted A\*](Search-14-WeightedAstar.ipynb) | A\* paresseux qui échange optimalité contre vitesse, même compromis que LDS |
| ↔ Python | [Search-13 — LDS (Python)](Search-13-LimitedDiscrepancySearch.ipynb) | Même algorithme en Python (jumeau) — comparer l'expressivité des deux langages |
| ↔ Sudoku | [Sudoku-1 — Backtracking](../../Sudoku/Sudoku-1-Backtracking-Csharp.ipynb) | Backtracking C# pur — LDS est une DFS bornée par le nombre d'écarts |

> **Référence :** Harvey & Ginsberg, *Limited Discrepancy Search*, IJCAI 1995.


## 6. Exercices

### Exercice 1 : Iterative LDS (ILDC)
Implémentez **ILDS** : exécutez LDS pour k=0, 1, 2, … en vous arrêtant dès que la
valeur **stagne** (deux incréments consécutifs sans amélioration) ou que k atteint n.
Comparez le nombre total de nœuds explorés par ILDS à l'énumération exhaustive sur
cette instance.
- **Indice :** le critère d'arrêt « valeur qui stagne » évite de gaspiller des nœuds
  pour des k qui n'apportent plus rien. Accumulez les nœuds de chaque appel `LdsKnapsack(k)`.


In [6]:
// Exercice 1 à compléter
// Conseil : bouclez k = 0..n, accumulez les nœuds, arrêtez quand la valeur cesse de croître.
// Comparez le total accumulé à exNodes (l'exhaustif calculé ci-dessus).
int totalNodesIlds = 0;
Console.WriteLine("Exercice 1 à compléter");


Exercice 1 à compléter


### Exercice 2 : passage à l'échelle — LDS vs exhaustif
Construisez une instance plus grande (par ex. n=18 objets aléatoires, capacité ~50%
du poids total) où le greedy est sous-optimal. Mesurez :
1. Le **plus petit k** pour lequel LDS atteint l'optimum (vérifié via DP ou exhaustif si n petit).
2. Le **ratio de nœuds** LDS(k\*) / exhaustif ($2^{18} = 262\,144$ feuilles).
- **Indice :** tirez des (poids, valeur) aléatoires avec une graine fixée (`new Random(42)`),
  calculez l'optimum exhaustif, puis balayez k jusqu'à l'atteindre.


In [7]:
// Exercice 2 à compléter
// Conseil : générez n=18 objets aléatoires (poids, valeur), calculez l'optimum exhaustif,
// puis balayez k jusqu'à l'atteindre. Affichez le ratio LDS(k*)/2**18.
Console.WriteLine("Exercice 2 à compléter");


Exercice 2 à compléter


### Exercice 3 : sensibilité à la qualité de l'heuristique
LDS suppose une heuristique *bonne*. Refaites le balayage k=0..n mais en **perturbant
l'ordre** des objets : (a) ordre aléatoire, (b) ordre par valeur absolue décroissante
(au lieu du ratio). Pour chaque heuristique, notez le **k minimum** pour atteindre
l'optimum.
- **Indice :** plus l'heuristique est mauvaise, plus k\* grandit — LDS perd tout son
  intérêt si l'heuristique est essentiellement aléatoire. Permutez le tableau `items`
  et réutilisez `LdsKnapsack`.


In [8]:
// Exercice 3 à compléter
// Conseil : permutez `items` selon différents ordres, et pour chacun cherchez le k min.
// Perturbation : utilisez new Random(123) pour un ordre aléatoire reproductible.
Console.WriteLine("Exercice 3 à compléter");


Exercice 3 à compléter


## Conclusion

**Limited Discrepancy Search** occupe une niche précise dans le spectre de la
recherche heuristique. À une extrémité, le **greedy** est instantané mais myope ;
à l'autre, la **DFS exhaustive** est exacte mais exponentielle. LDS exploite l'idée
que *si l'heuristique est bonne, l'optimum est un proche voisin du greedy* : en
bornant le nombre d'écarts $k$, on obtient un compromis réglable entre qualité et
coût.

### Comparaison C# vs Python (jumeau)

Le port C# met en évidence :
- le **typage statique** des `record Item` (sécurité compile-time vs tuples dynamiques Python) ;
- la **récursion avec closure** (`void Rec(...) { ... }` capturant `bestValue`, `nodes`)
  vs la fermeture Python `def rec(...)` avec `nonlocal` ;
- la **manipulation de bits** pour l'exhaustif (`1L << n`, `(mask >> i) & 1L`) plus
  explicite en C# qu'avec `itertools.product`.

L'algorithmique, elle, est **identique** — LDS est language-agnostic. Voir le
[jumeau Python](Search-13-LimitedDiscrepancySearch.ipynb) pour la version dynamique.

> **Référence :** Harvey, W. D., & Ginsberg, M. L. (1995). *Limited Discrepancy Search.*
> Proceedings of IJCAI 1995.
